# LeetCode 633 — Sum of Square Numbers bằng Python

Link đề: https://leetcode.com/problems/sum-of-square-numbers/ · Độ khó: **Medium** · Chủ đề: Math, Two Pointers, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def judgeSquareSum(self, c: int) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho số nguyên không âm `c`, quyết định có hai số nguyên `a`, `b` sao cho `a^2 + b^2 = c` hay không (`a` và `b` được phép bằng `0` và được phép bằng nhau).
2. Nắm các công cụ Python: `math.isqrt` (căn bậc hai nguyên **chính xác**), kiểm tra số chính phương, hai con trỏ, tìm kiếm nhị phân tự viết, phân tích ra thừa số nguyên tố bằng chia thử, toán tử `%` và `//`.
3. Đi từ **vét cạn hai vòng lặp** → **cố định `a`, tính `b` bằng `isqrt`** → **hai con trỏ** → **tìm nhị phân cho `b`** → **định lý Fermat về tổng hai bình phương (phân tích thừa số)**.
4. Hiểu mấu chốt: khi đã chọn `a` thì `b^2 = c - a^2` **được xác định**, nên ta chỉ cần duyệt **một** biến (`O(sqrt c)` thay vì `O(c)`); và `a <= b` nên `a` chỉ cần chạy đến `sqrt(c/2)`.
5. Tránh các lỗi kinh điển: quên trường hợp `a == b` hoặc `c = 0`, quên `+1` ở cận của `range`, dùng `sqrt` số thực để so sánh bằng, vét cạn `O(c)` bị TLE, và khi dùng định lý Fermat thì quên thừa số nguyên tố còn dư sau vòng chia thử hoặc quên xét số mũ chẵn.

## 2. Đề bài & ví dụ

Cho một số nguyên không âm `c`. Hãy quyết định có tồn tại hai số nguyên `a` và `b` sao cho `a^2 + b^2 = c` hay không. Trả về `True` nếu có, `False` nếu không.

**Ví dụ 1**

- Input: `c = 5`
- Output: `true`
- Giải thích: `1 * 1 + 2 * 2 = 5`.

**Ví dụ 2**

- Input: `c = 3`
- Output: `false`
- Giải thích: các bình phương không vượt quá 3 là `0, 1`; không có hai số nào (kể cả lặp) cộng ra 3.

**Ràng buộc:**

- `0 <= c <= 2^31 - 1`

Chú ý: `a`, `b` có thể bằng `0` (nên `c = 4 = 0^2 + 2^2` đúng) và có thể bằng nhau (nên `c = 2 = 1^2 + 1^2` đúng). Số âm không giúp ích gì thêm vì `(-a)^2 = a^2`, nên ta chỉ cần xét `a, b >= 0`.

In [1]:
from math import isqrt, sqrt
from bisect import bisect_left
import random
import time

examples = [(5, True), (3, False)]
for c, expected in examples:
    print(f"c = {c} -> kỳ vọng {expected}")
print("căn bậc hai của giới hạn trên:", isqrt(2**31 - 1))

c = 5 -> kỳ vọng True
c = 3 -> kỳ vọng False
căn bậc hai của giới hạn trên: 46340


## 3. Kiến thức nền

### 3.1. Số chính phương và `math.isqrt`

**Số chính phương** là bình phương của một số nguyên: `0, 1, 4, 9, 16, 25, ...`. `math.isqrt(n)` trả về **phần nguyên của căn bậc hai** của `n` bằng thuật toán trên số nguyên, nên **luôn chính xác** (không có sai số làm tròn như `sqrt` số thực). `n` là số chính phương ⇔ `isqrt(n) ** 2 == n`.

In [2]:
for n in (0, 1, 2, 15, 16, 17, 25, 26):
    r = isqrt(n)
    print(f"isqrt({n:>2}) = {r}  | {r}^2 = {r * r:>2}  | {n} là số chính phương? {r * r == n}")

isqrt( 0) = 0  | 0^2 =  0  | 0 là số chính phương? True
isqrt( 1) = 1  | 1^2 =  1  | 1 là số chính phương? True
isqrt( 2) = 1  | 1^2 =  1  | 2 là số chính phương? False
isqrt(15) = 3  | 3^2 =  9  | 15 là số chính phương? False
isqrt(16) = 4  | 4^2 = 16  | 16 là số chính phương? True
isqrt(17) = 4  | 4^2 = 16  | 17 là số chính phương? False
isqrt(25) = 5  | 5^2 = 25  | 25 là số chính phương? True
isqrt(26) = 5  | 5^2 = 25  | 26 là số chính phương? False


### 3.2. Vì sao tránh `sqrt` số thực khi so sánh bằng

`math.sqrt` trả về `float` (số thực 64-bit); với số rất lớn, căn bậc hai có thể bị sai ở chữ số cuối cùng, nên `sqrt(n) == int(sqrt(n))` có thể sai. Với `n < 2^31` thì `float` còn chính xác, nhưng thói quen an toàn là dùng `isqrt`. Dưới đây là một ví dụ cho thấy `float` mất độ chính xác ở số lớn (ngoài ràng buộc của bài):

In [3]:
n = (10**8 + 1) ** 2 - 1        # = 10^16 + 2*10^8: ngay dưới một số chính phương lớn
print("n =", n)
print("isqrt(n)      =", isqrt(n), "(đúng: 100000000)")
print("int(sqrt(n))  =", int(sqrt(n)), "<- SAI: float làm tròn lên, đúng phải là 100000000")
big = 10**30 + 1
print("isqrt(10^30 + 1) =", isqrt(big))
print("sqrt(10^30 + 1)  =", sqrt(big))

n = 10000000200000000
isqrt(n)      = 100000000 (đúng: 100000000)
int(sqrt(n))  = 100000001 <- SAI: float làm tròn lên, đúng phải là 100000000
isqrt(10^30 + 1) = 1000000000000000
sqrt(10^30 + 1)  = 1000000000000000.0


Python có số nguyên **không giới hạn**, nên `a * a` không bao giờ tràn (khác với C++/Java, nơi `a * a` với `a` cỡ `46340` có thể tràn `int` nếu cộng thêm). Dù vậy, hiểu giới hạn `2^31 - 1` giúp ta ước lượng thời gian: `sqrt(2^31 - 1) ≈ 46340`.

### 3.3. Vòng lặp `while` với điều kiện kết thúc

Ta sẽ dùng `while a <= b` cho hai con trỏ, và `while 2 * a * a <= c` cho vòng duyệt `a`. Nhớ: điều kiện đúng thì **chạy tiếp**, sai thì dừng.

In [4]:
a = 0
while 2 * a * a <= 10:
    print(f"a = {a}: 2a^2 = {2 * a * a} <= 10 -> còn chạy")
    a += 1
print("dừng vì 2a^2 =", 2 * a * a, "> 10")

a = 0: 2a^2 = 0 <= 10 -> còn chạy
a = 1: 2a^2 = 2 <= 10 -> còn chạy
a = 2: 2a^2 = 8 <= 10 -> còn chạy
dừng vì 2a^2 = 18 > 10


### 3.4. Hai con trỏ trên "dãy các bình phương"

Dãy `0^2, 1^2, 2^2, ..., (isqrt c)^2` là dãy **tăng dần**. Ta đặt một con trỏ `a` ở đầu và một con trỏ `b` ở cuối rồi xét tổng `a^2 + b^2`: nếu **nhỏ hơn** `c` thì tăng `a` (cần số lớn hơn); nếu **lớn hơn** thì giảm `b`. Đây giống bài Two Sum trên mảng đã sắp xếp, chỉ khác là "mảng" ở đây là dãy bình phương không cần lưu trong bộ nhớ (tính `a * a` khi cần).

### 3.5. Tìm kiếm nhị phân

Tìm kiếm nhị phân tìm một giá trị trong dãy **đã sắp xếp** bằng cách luôn kiểm tra phần tử giữa và loại bỏ một nửa. Với hàm `f(b) = b * b` **đơn điệu tăng**, ta có thể tìm `b` sao cho `b * b == t` trong `O(log t)` bước mà không cần `isqrt`.

In [5]:
def has_square_root(t):
    """Tìm nhị phân b trong [0, t] sao cho b*b == t."""
    lo, hi = 0, t
    while lo <= hi:
        mid = (lo + hi) // 2
        sq = mid * mid
        if sq == t:
            return True
        if sq < t:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

print([t for t in range(0, 40) if has_square_root(t)])

[0, 1, 4, 9, 16, 25, 36]


### 3.6. Phân tích thừa số nguyên tố bằng chia thử, và phép `%`

Để phân tích `c = p1^e1 * p2^e2 * ...`, ta thử chia cho `2, 3, 4, ...` (các hợp số tự động không chia được vì thừa số nhỏ hơn của chúng đã bị loại hết). Chỉ cần thử `i` đến khi `i * i > c`; phần còn lại (nếu `> 1`) là **một số nguyên tố**. Phép `p % 4` cho biết số nguyên tố lẻ `p` có dạng `4k + 1` hay `4k + 3`; ta sẽ cần điều này ở cách 5.

In [6]:
def factorize(n):
    result = {}
    i = 2
    while i * i <= n:
        while n % i == 0:
            result[i] = result.get(i, 0) + 1
            n //= i
        i += 1
    if n > 1:
        result[n] = result.get(n, 0) + 1
    return result

for n in (45, 21, 12, 50, 7, 1, 1000):
    f = factorize(n)
    print(f"{n:>4} = {f}  | các nguyên tố lẻ theo mod 4: {[(p, p % 4) for p in f if p % 2 == 1]}")

  45 = {3: 2, 5: 1}  | các nguyên tố lẻ theo mod 4: [(3, 3), (5, 1)]
  21 = {3: 1, 7: 1}  | các nguyên tố lẻ theo mod 4: [(3, 3), (7, 3)]
  12 = {2: 2, 3: 1}  | các nguyên tố lẻ theo mod 4: [(3, 3)]
  50 = {2: 1, 5: 2}  | các nguyên tố lẻ theo mod 4: [(5, 1)]
   7 = {7: 1}  | các nguyên tố lẻ theo mod 4: [(7, 3)]
   1 = {}  | các nguyên tố lẻ theo mod 4: []
1000 = {2: 3, 5: 3}  | các nguyên tố lẻ theo mod 4: [(5, 1)]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

**`c = 5`:** các bình phương không vượt quá 5 là `0, 1, 4`. Thử từng `a`:

| `a` | `a^2` | `c - a^2` | có phải số chính phương? |
|---|---|---|---|
| 0 | 0 | 5 | không |
| 1 | 1 | 4 | **có** (`2^2`) → `1^2 + 2^2 = 5` |

**`c = 3`:** `a = 0` → `3` (không); `a = 1` → `2` (không). Hết `a` (vì `2^2 = 4 > 3`) nên đáp án `False`.

**`c = 25`:** `a = 0` → `25 = 5^2` ✓ (`0 + 25`); cũng `a = 3` → `16 = 4^2` ✓ (`9 + 16`). Có nhiều đáp án, ta chỉ cần **một**.

**Nhận xét quan trọng:** khi đã chọn `a`, số `b^2 = c - a^2` **hoàn toàn xác định**, nên ta không cần thử mọi `b`; chỉ cần kiểm tra `c - a^2` có phải **số chính phương** hay không. Hơn nữa, nếu `(a, b)` là đáp án thì `(b, a)` cũng vậy, nên ta có thể giả sử `a <= b`, tức `2a^2 <= c`.

In [7]:
c = 25
print(f"c = {c}")
for a in range(isqrt(c) + 1):
    rest = c - a * a
    b = isqrt(rest)
    print(f"a = {a}: c - a^2 = {rest:>2}, isqrt = {b}, {b}^2 = {b * b:>2} -> {'ĐÚNG, (a, b) = ' + str((a, b)) if b * b == rest else 'không'}")

c = 25
a = 0: c - a^2 = 25, isqrt = 5, 5^2 = 25 -> ĐÚNG, (a, b) = (0, 5)
a = 1: c - a^2 = 24, isqrt = 4, 4^2 = 16 -> không
a = 2: c - a^2 = 21, isqrt = 4, 4^2 = 16 -> không
a = 3: c - a^2 = 16, isqrt = 4, 4^2 = 16 -> ĐÚNG, (a, b) = (3, 4)
a = 4: c - a^2 =  9, isqrt = 3, 3^2 =  9 -> ĐÚNG, (a, b) = (4, 3)
a = 5: c - a^2 =  0, isqrt = 0, 0^2 =  0 -> ĐÚNG, (a, b) = (5, 0)


## 5. Cách 1: Brute Force (vét cạn)

### Ý tưởng

Làm đúng theo định nghĩa: thử **mọi cặp** `(a, b)` với `0 <= a <= b <= isqrt(c)` và kiểm tra `a * a + b * b == c`. Ta chỉ cần xét `b <= isqrt(c)` vì `b^2 <= c`.

In [8]:
def judge_brute(c):
    r = isqrt(c)
    for a in range(r + 1):
        for b in range(a, r + 1):
            if a * a + b * b == c:
                return True
    return False


for c, expected in examples:
    got = judge_brute(c)
    print(c, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

5 -> True (kỳ vọng True )
3 -> False (kỳ vọng False )


### Độ phức tạp

- Số cặp `(a, b)` là cỡ `(sqrt c)^2 / 2 = c / 2`. Với `c = 2^31 - 1` là khoảng **1,07 tỉ** phép thử — Python chỉ làm được vài chục triệu phép như vậy mỗi giây ⇒ cỡ cả phút, **quá chậm**.
- Time `O(c)`; Space `O(1)`.

### Nút thắt

Vòng lặp trong cùng thử rất nhiều `b` không cần thiết: với mỗi `a`, **chỉ có đúng một `b` có thể đúng** (`b = sqrt(c - a^2)`). Hãy đếm số phép thử:

In [9]:
def judge_brute_counting(c):
    r = isqrt(c)
    tries = 0
    for a in range(r + 1):
        for b in range(a, r + 1):
            tries += 1
            if a * a + b * b == c:
                return True, tries
    return False, tries

for c in (3, 99, 999, 9_999, 99_999):      # chọn c không biểu diễn được -> phải duyệt hết
    ok, tries = judge_brute_counting(c)
    print(f"c = {c:>6}: {ok!s:5} sau {tries:>9,} phép thử | c/2 = {c // 2:>7,}")

c =      3: False sau         3 phép thử | c/2 =       1
c =     99: False sau        55 phép thử | c/2 =      49
c =    999: False sau       528 phép thử | c/2 =     499
c =   9999: False sau     5,050 phép thử | c/2 =   4,999
c =  99999: False sau    50,403 phép thử | c/2 =  49,999


Số phép thử bám sát `c / 2`. Với `c` cỡ `2 * 10^9` thì không thể chạy. Ta cần chọn `a`, rồi **tra** `b`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Cố định `a` thì `b` được xác định:** `b^2 = c - a^2`. Chỉ cần kiểm tra `c - a^2` có phải số chính phương không (bằng `isqrt` trong gần `O(1)`).
2. **Giả sử `a <= b`:** nếu `a^2 + b^2 = c` thì `2a^2 <= a^2 + b^2 = c`, nên chỉ cần duyệt `a` từ `0` đến `isqrt(c // 2)`. Với `c = 2^31 - 1` đó chỉ là khoảng **32.768** giá trị `a`.

Vậy ta đã giảm từ `O(c)` xuống `O(sqrt c)`. Từ cùng ý tưởng này sinh ra nhiều cách:

- **Cách 2:** duyệt `a`, kiểm tra `c - a^2` là số chính phương bằng `isqrt`.
- **Cách 3:** hai con trỏ `a` (tăng) và `b` (giảm) trên dãy bình phương.
- **Cách 4:** duyệt `a`, **tìm nhị phân** `b` thay cho `isqrt`.
- **Cách 5:** dùng **định lý Fermat về tổng hai bình phương** để quyết định chỉ bằng cách phân tích `c` ra thừa số nguyên tố.

In [10]:
# minh họa: với c lớn nhất, số giá trị a cần thử khi giả sử a <= b
c = 2**31 - 1
print("duyệt a đến isqrt(c//2) =", isqrt(c // 2), "giá trị (so với c/2 = %d phép của vét cạn)" % (c // 2))

duyệt a đến isqrt(c//2) = 32767 giá trị (so với c/2 = 1073741823 phép của vét cạn)


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt `a`, kiểm tra `c - a^2` có phải số chính phương

**Giải thích từng đoạn:**

1. `a = 0`; lặp trong khi `2 * a * a <= c` (đảm bảo `a <= b`).
2. Với mỗi `a`: `rest = c - a * a`, `b = isqrt(rest)`.
3. Nếu `b * b == rest` thì `rest` là số chính phương → trả `True`.
4. Hết vòng lặp mà chưa thấy → `False`.

**Code demo:**

In [11]:
def judge_isqrt_demo(c):
    a = 0
    while 2 * a * a <= c:
        rest = c - a * a
        b = isqrt(rest)
        print(f"  a = {a}: rest = {rest}, isqrt = {b}, {b}*{b} = {b * b}")
        if b * b == rest:
            print(f"  => {a}^2 + {b}^2 = {c}")
            return True
        a += 1
    return False

print(judge_isqrt_demo(5))
print(judge_isqrt_demo(3))

  a = 0: rest = 5, isqrt = 2, 2*2 = 4
  a = 1: rest = 4, isqrt = 2, 2*2 = 4
  => 1^2 + 2^2 = 5
True
  a = 0: rest = 3, isqrt = 1, 1*1 = 1
  a = 1: rest = 2, isqrt = 1, 1*1 = 1
False


**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    def judgeSquareSum(self, c: int) -> bool:
        a = 0
        while 2 * a * a <= c:
            rest = c - a * a
            b = isqrt(rest)
            if b * b == rest:
                return True
            a += 1
        return False


for c, expected in examples:
    assert Solution2().judgeSquareSum(c) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** Time `O(sqrt c)` (tối đa khoảng `sqrt(c / 2)` vòng, mỗi vòng `isqrt` rất nhanh với số nhỏ). Space `O(1)`.

### 7.2. Cách 3 — Hai con trỏ

**Ý tưởng khác:** thay vì tính `b` từ `a`, giữ **cả hai** con trỏ: `a = 0`, `b = isqrt(c)`. Xét `s = a^2 + b^2`:

- `s == c` → tìm được.
- `s < c` → tổng còn thiếu, tăng `a`.
- `s > c` → tổng đang dư, giảm `b`.
- Dừng khi `a > b` (hai con trỏ vượt qua nhau) → `False`.

**Giải thích từng đoạn:**

1. Khởi tạo `a = 0`, `b = isqrt(c)`. (`b` không thể lớn hơn thế vì `b^2 <= c`.)
2. `while a <= b`: cho phép `a == b` (ví dụ `c = 2 = 1^2 + 1^2`).
3. Mỗi lần lặp hoặc `a` tăng hoặc `b` giảm nên tổng số bước tối đa là `(isqrt c) + 1`.

**Code demo:**

In [13]:
def two_pointers_demo(c):
    a, b = 0, isqrt(c)
    while a <= b:
        s = a * a + b * b
        print(f"  a = {a}, b = {b}: {a}^2 + {b}^2 = {s} {'==' if s == c else ('<' if s < c else '>')} {c}")
        if s == c:
            return True
        if s < c:
            a += 1
        else:
            b -= 1
    return False

print(two_pointers_demo(5))
print(two_pointers_demo(3))

  a = 0, b = 2: 0^2 + 2^2 = 4 < 5
  a = 1, b = 2: 1^2 + 2^2 = 5 == 5
True
  a = 0, b = 1: 0^2 + 1^2 = 1 < 3
  a = 1, b = 1: 1^2 + 1^2 = 2 < 3
False


**Code nộp LeetCode (cách 3):**

In [14]:
class Solution3:
    def judgeSquareSum(self, c: int) -> bool:
        a, b = 0, isqrt(c)
        while a <= b:
            s = a * a + b * b
            if s == c:
                return True
            if s < c:
                a += 1
            else:
                b -= 1
        return False


for c, expected in examples:
    assert Solution3().judgeSquareSum(c) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** Time `O(sqrt c)` (mỗi vòng thu hẹp `[a, b]` đi 1 đơn vị; độ rộng ban đầu `isqrt(c) + 1`). Space `O(1)`.

Vì sao không bỏ sót đáp án? Mục 10 chứng minh rằng nếu có đáp án `(a*, b*)` thì hai con trỏ không bao giờ "nhảy qua" nó.

### 7.3. Cách 4 — Duyệt `a`, tìm **nhị phân** `b`

**Ý tưởng khác:** với mỗi `a`, ta cần biết `c - a^2` có phải số chính phương không nhưng **không dùng `isqrt`** — tự tìm nhị phân `b` trong `[a, isqrt(c)]` sao cho `b * b == c - a * a`. Hàm `b * b` đơn điệu tăng nên tìm nhị phân hợp lệ.

**Giải thích từng đoạn:**

1. Vòng ngoài: `a` từ `0` khi `2a^2 <= c`.
2. Đặt `target = c - a * a`; tìm trong `[a, ...]` bằng `lo, hi`.
3. Mỗi bước kiểm tra `mid * mid` so với `target`; bỏ nửa không chứa đáp án.

**Code demo:**

In [15]:
def binary_search_demo(c):
    a = 0
    while 2 * a * a <= c:
        target = c - a * a
        lo, hi = a, target            # b nằm trong [a, target]; chặn trên thô cũng được
        steps = 0
        found = False
        while lo <= hi:
            mid = (lo + hi) // 2
            steps += 1
            sq = mid * mid
            if sq == target:
                found = True
                break
            if sq < target:
                lo = mid + 1
            else:
                hi = mid - 1
        print(f"  a = {a}: tìm b với b^2 = {target} trong {steps} bước -> {'b = ' + str(mid) if found else 'không có'}")
        if found:
            return True
        a += 1
    return False

print(binary_search_demo(13))
print(binary_search_demo(11))

  a = 0: tìm b với b^2 = 13 trong 4 bước -> không có
  a = 1: tìm b với b^2 = 12 trong 3 bước -> không có
  a = 2: tìm b với b^2 = 9 trong 2 bước -> b = 3
True
  a = 0: tìm b với b^2 = 11 trong 4 bước -> không có
  a = 1: tìm b với b^2 = 10 trong 4 bước -> không có
  a = 2: tìm b với b^2 = 7 trong 3 bước -> không có
False


**Code nộp LeetCode (cách 4):**

In [16]:
class Solution4:
    def judgeSquareSum(self, c: int) -> bool:
        a = 0
        while 2 * a * a <= c:
            target = c - a * a
            lo, hi = a, isqrt(target) + 1       # b <= isqrt(target); +1 cho an toàn
            while lo <= hi:
                mid = (lo + hi) // 2
                sq = mid * mid
                if sq == target:
                    return True
                if sq < target:
                    lo = mid + 1
                else:
                    hi = mid - 1
            a += 1
        return False


for c, expected in examples:
    assert Solution4().judgeSquareSum(c) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp:** Time `O(sqrt(c) * log c)` (với mỗi trong `~sqrt(c / 2)` giá trị `a`, một lần tìm nhị phân tốn `O(log c)`). Space `O(1)`. Chậm hơn cách 2 một hệ số `log`, nhưng minh họa cách thay `isqrt` bằng tìm nhị phân (hữu ích ở ngôn ngữ không có `isqrt` chính xác).

(Dòng `hi = isqrt(target) + 1` ở bản nộp chỉ để thu hẹp khoảng tìm; có dùng `isqrt` nhưng kiểm tra bằng nhị phân vẫn đúng. Bản demo ở trên dùng `hi = target` để cho thấy nhị phân thuần túy.)

### 7.4. Cách 5 — Định lý Fermat về tổng hai bình phương (toán học)

**Định lý (Fermat–Euler, tổng hai bình phương):** một số nguyên dương `c` biểu diễn được dưới dạng `a^2 + b^2` **khi và chỉ khi** trong phân tích ra thừa số nguyên tố của `c`, **mọi số nguyên tố có dạng `4k + 3` đều xuất hiện với số mũ chẵn**. (Số `0` biểu diễn được: `0 + 0`.)

Ví dụ:

- `45 = 3^2 * 5`: nguyên tố `3 ≡ 3 (mod 4)` có số mũ `2` (chẵn) → `True` (`36 + 9`).
- `21 = 3 * 7`: `3` mũ `1` (lẻ) → `False`.
- `12 = 2^2 * 3`: `3` mũ `1` (lẻ) → `False`.
- `50 = 2 * 5^2`: không có nguyên tố dạng `4k + 3` → `True` (`1 + 49` hay `25 + 25`).

Ta không chứng minh định lý ở đây (cần số học Gauss); thay vào đó ta **kiểm chứng bằng vét cạn** ở mục 12 cho mọi `c` đến vài chục nghìn.

**Giải thích từng đoạn:**

1. Với `i` từ `2` khi `i * i <= c`: nếu `i` chia hết `c`, đếm số mũ `e` bằng cách chia liên tiếp.
2. Nếu `i % 4 == 3` và `e` lẻ → trả `False` ngay.
3. Sau vòng lặp, nếu `c > 1` thì phần còn lại là **một số nguyên tố** với mũ 1. Nếu nó `≡ 3 (mod 4)` thì `False`.
4. Còn lại → `True`. Trường hợp `c = 0` rơi vào đây (vòng lặp không chạy, `0 % 4 != 3`) nên trả `True` đúng.

**Code demo:**

In [17]:
def fermat_demo(c):
    n = c
    i = 2
    while i * i <= n:
        if n % i == 0:
            e = 0
            while n % i == 0:
                n //= i
                e += 1
            print(f"  nguyên tố {i} (mod 4 = {i % 4}) mũ {e}")
            if i % 4 == 3 and e % 2 == 1:
                print("  => nguyên tố 4k+3 với mũ lẻ -> False")
                return False
        i += 1
    if n > 1:
        print(f"  còn dư nguyên tố {n} (mod 4 = {n % 4}) mũ 1")
    return n % 4 != 3

for c in (45, 21, 12, 50, 7):
    print(c, "->", fermat_demo(c))

  nguyên tố 3 (mod 4 = 3) mũ 2
  còn dư nguyên tố 5 (mod 4 = 1) mũ 1
45 -> True
  nguyên tố 3 (mod 4 = 3) mũ 1
  => nguyên tố 4k+3 với mũ lẻ -> False
21 -> False
  nguyên tố 2 (mod 4 = 2) mũ 2
  còn dư nguyên tố 3 (mod 4 = 3) mũ 1
12 -> False
  nguyên tố 2 (mod 4 = 2) mũ 1
  nguyên tố 5 (mod 4 = 1) mũ 2
50 -> True
  còn dư nguyên tố 7 (mod 4 = 3) mũ 1
7 -> False


**Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    def judgeSquareSum(self, c: int) -> bool:
        i = 2
        while i * i <= c:
            if c % i == 0:
                e = 0
                while c % i == 0:
                    c //= i
                    e += 1
                if i % 4 == 3 and e % 2 == 1:
                    return False
            i += 1
        return c % 4 != 3


for c, expected in examples:
    assert Solution5().judgeSquareSum(c) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp:** Time `O(sqrt c)` trong trường hợp xấu (khi `c` là số nguyên tố lớn, vòng `i` phải chạy đến `sqrt c`); thường nhanh hơn nhiều khi `c` có thừa số nhỏ vì `c` bị rút gọn dần (và điều kiện `i * i <= c` dùng giá trị `c` đã rút gọn). Space `O(1)`. Lưu ý: dòng `c //= i` làm thay đổi biến `c` — đây là chủ ý, vòng lặp luôn so với `c` đã rút gọn.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (duyệt `a`, `isqrt`)

Bảng trạng thái với `c = 99` (không biểu diễn được: `99 = 9 * 11 = 3^2 * 11`, `11 ≡ 3 mod 4` mũ lẻ):

| Cột | Ý nghĩa |
|---|---|
| `a` | giá trị đang thử |
| `rest` | `c - a^2` |
| `b = isqrt(rest)` | ứng viên của `b` |
| `b^2 == rest?` | có phải số chính phương |

In [19]:
def trace_isqrt(c):
    print(f"c = {c}, giới hạn a: 2a^2 <= {c} -> a <= {isqrt(c // 2)}")
    a = 0
    while 2 * a * a <= c:
        rest = c - a * a
        b = isqrt(rest)
        ok = b * b == rest
        print(f"a = {a:>2} | rest = {rest:>3} | b = {b:>2} | b^2 = {b * b:>3} | {'ĐÚNG' if ok else 'không'}")
        if ok:
            return True
        a += 1
    return False

print("kết quả:", trace_isqrt(99))
print("kết quả:", trace_isqrt(58))

c = 99, giới hạn a: 2a^2 <= 99 -> a <= 7
a =  0 | rest =  99 | b =  9 | b^2 =  81 | không
a =  1 | rest =  98 | b =  9 | b^2 =  81 | không
a =  2 | rest =  95 | b =  9 | b^2 =  81 | không
a =  3 | rest =  90 | b =  9 | b^2 =  81 | không
a =  4 | rest =  83 | b =  9 | b^2 =  81 | không
a =  5 | rest =  74 | b =  8 | b^2 =  64 | không
a =  6 | rest =  63 | b =  7 | b^2 =  49 | không
a =  7 | rest =  50 | b =  7 | b^2 =  49 | không
kết quả: False
c = 58, giới hạn a: 2a^2 <= 58 -> a <= 5
a =  0 | rest =  58 | b =  7 | b^2 =  49 | không
a =  1 | rest =  57 | b =  7 | b^2 =  49 | không
a =  2 | rest =  54 | b =  7 | b^2 =  49 | không
a =  3 | rest =  49 | b =  7 | b^2 =  49 | ĐÚNG
kết quả: True


Quan sát: với `c = 58` ta thấy `a = 3` cho `rest = 49 = 7^2` → `9 + 49 = 58`. Với `c = 99`, mọi `rest` đều rơi giữa hai số chính phương liên tiếp.

### 8.2. Trace cách 3 (hai con trỏ)

Mỗi dòng cho thấy `a`, `b`, tổng và quyết định di chuyển con trỏ:

In [20]:
def trace_two_pointers(c):
    a, b = 0, isqrt(c)
    print(f"c = {c}: bắt đầu a = {a}, b = {b}")
    step = 0
    while a <= b:
        s = a * a + b * b
        step += 1
        if s == c:
            print(f"bước {step}: {a}^2 + {b}^2 = {s} == c -> ĐÚNG")
            return True
        if s < c:
            print(f"bước {step}: {a}^2 + {b}^2 = {s} < c -> a += 1")
            a += 1
        else:
            print(f"bước {step}: {a}^2 + {b}^2 = {s} > c -> b -= 1")
            b -= 1
    print(f"a = {a} > b = {b} -> hết, không có đáp án")
    return False

trace_two_pointers(58)
trace_two_pointers(14)

c = 58: bắt đầu a = 0, b = 7
bước 1: 0^2 + 7^2 = 49 < c -> a += 1
bước 2: 1^2 + 7^2 = 50 < c -> a += 1
bước 3: 2^2 + 7^2 = 53 < c -> a += 1
bước 4: 3^2 + 7^2 = 58 == c -> ĐÚNG
c = 14: bắt đầu a = 0, b = 3
bước 1: 0^2 + 3^2 = 9 < c -> a += 1
bước 2: 1^2 + 3^2 = 10 < c -> a += 1
bước 3: 2^2 + 3^2 = 13 < c -> a += 1
bước 4: 3^2 + 3^2 = 18 > c -> b -= 1
a = 3 > b = 2 -> hết, không có đáp án


False

### 8.3. Trace cách 5 (Fermat)

Ta in từng bước chia thử và trạng thái `c` còn lại, với `c = 360 = 2^3 * 3^2 * 5` (đáp án `True`: `360 = 6^2 + 18^2`) và `c = 252 = 2^2 * 3^2 * 7` (đáp án `False` vì `7 ≡ 3 mod 4` mũ 1).

In [21]:
def trace_fermat(c0):
    c = c0
    print(f"c = {c0}")
    i = 2
    while i * i <= c:
        if c % i == 0:
            e = 0
            while c % i == 0:
                c //= i
                e += 1
            verdict = ""
            if i % 4 == 3:
                verdict = " | mũ lẻ -> LOẠI" if e % 2 == 1 else " | mũ chẵn -> ổn"
            print(f"  i = {i}: chia {e} lần, c còn {c}{verdict}")
            if i % 4 == 3 and e % 2 == 1:
                return False
        i += 1
    print(f"  dừng vì i*i = {i * i} > c = {c}")
    if c > 1:
        print(f"  còn dư nguyên tố {c}, mod 4 = {c % 4}")
    return c % 4 != 3

print("kết quả:", trace_fermat(360))
print("kết quả:", trace_fermat(252))

c = 360
  i = 2: chia 3 lần, c còn 45
  i = 3: chia 2 lần, c còn 5 | mũ chẵn -> ổn
  dừng vì i*i = 16 > c = 5
  còn dư nguyên tố 5, mod 4 = 1
kết quả: True
c = 252
  i = 2: chia 2 lần, c còn 63
  i = 3: chia 2 lần, c còn 7 | mũ chẵn -> ổn
  dừng vì i*i = 16 > c = 7
  còn dư nguyên tố 7, mod 4 = 3
kết quả: False


## 9. Code nộp LeetCode

Chọn **cách 3 (hai con trỏ)**: ngắn, `O(sqrt c)`, `O(1)` bộ nhớ, đúng với mọi `c` trong ràng buộc và là cách phổ biến nhất. Đây là code đúng class/method/signature:

In [22]:
from math import isqrt


class Solution:
    def judgeSquareSum(self, c: int) -> bool:
        a, b = 0, isqrt(c)
        while a <= b:
            s = a * a + b * b
            if s == c:
                return True
            if s < c:
                a += 1
            else:
                b -= 1
        return False


for c, expected in examples:
    got = Solution().judgeSquareSum(c)
    print(c, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

5 -> True (kỳ vọng True )
3 -> False (kỳ vọng False )


## 10. Vì sao thuật toán đúng

**Cách 2.** Nếu tồn tại `(a, b)` với `a^2 + b^2 = c` thì tồn tại cặp với `0 <= a <= b` (đổi chỗ, và dùng giá trị tuyệt đối). Khi đó `2a^2 <= c`, nên `a` nằm trong khoảng vòng lặp duyệt. Với `a` đó, `rest = c - a^2 = b^2` là số chính phương nên `isqrt(rest) = b` và kiểm tra `b * b == rest` đúng → trả `True`. Ngược lại, nếu thuật toán trả `True` ở một `a` nào đó thì rõ ràng `a^2 + b^2 = c`. Vậy trả `True` ⇔ tồn tại biểu diễn.

**Cách 3 (hai con trỏ).** Nếu không có đáp án thì không bao giờ gặp `s == c`, vòng lặp kết thúc (mỗi bước `b - a` giảm 1) và trả `False`. Nếu có đáp án `(a*, b*)` với `0 <= a* <= b* <= isqrt(c)`, ta chứng minh **bất biến**: `a <= a*` và `b >= b*` luôn đúng cho đến khi gặp `s == c`.

- Lúc đầu `a = 0 <= a*` và `b = isqrt(c) >= b*`.
- Giả sử bất biến đúng và `s < c`, ta tăng `a`. Nếu `a = a*` thì `b > b*` (vì `b = b*` cho `s = c`, đã dừng), suy ra `s = a*^2 + b^2 > a*^2 + b*^2 = c`, mâu thuẫn với `s < c`. Vậy `a < a*`, và sau khi tăng vẫn `a <= a*`.
- Đối xứng, nếu `s > c` và ta giảm `b`: nếu `b = b*` thì `a < a*` nên `s < c`, mâu thuẫn; vậy `b > b*` và sau khi giảm vẫn `b >= b*`.

Bất biến cho `a <= a* <= b* <= b`, nên điều kiện `a <= b` luôn đúng, vòng lặp không thể kết thúc mà chưa gặp đáp án; vì mỗi bước `b - a` giảm 1 nên sau hữu hạn bước phải đạt `a = a*`, `b = b*` và `s == c`.

**Cách 4.** Giống cách 2, chỉ thay phép kiểm tra "số chính phương" bằng tìm nhị phân trên hàm đơn điệu `b * b`, đúng vì tìm nhị phân trên dãy tăng chặt tìm được `b` nếu tồn tại.

**Cách 5.** Dựa trên định lý Fermat–Euler: `c` là tổng hai bình phương ⇔ mọi nguyên tố `p ≡ 3 (mod 4)` có số mũ chẵn trong phân tích của `c`. Code phân tích `c` bằng chia thử: với mỗi nguyên tố `i` chia hết `c`, nó chia hết mọi lũy thừa của `i` rồi `c` bị rút gọn nên `i` sau đó không còn chia hết (các hợp số không xuất hiện). Phần dư sau vòng lặp chỉ có thể là một nguyên tố có mũ 1 (nếu còn hai nguyên tố chưa chia, tích của chúng nhỏ nhất đã `> i^2` vi phạm điều kiện dừng). Điều kiện cuối `c % 4 != 3` kiểm tra nguyên tố dư đó. `c = 0` và `c = 1` trả `True` (`0 = 0^2 + 0^2`, `1 = 0^2 + 1^2`); nguyên tố `2` không ảnh hưởng (`2 = 1^2 + 1^2`).

**Vét cạn.** Bản dịch nguyên văn của định nghĩa.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn hai vòng | `O(c)` | `O(1)` |
| 2. Duyệt `a`, `isqrt` | `O(sqrt c)` | `O(1)` |
| 3. Hai con trỏ (nộp) | `O(sqrt c)` | `O(1)` |
| 4. Duyệt `a`, tìm nhị phân `b` | `O(sqrt(c) * log c)` | `O(1)` |
| 5. Fermat (phân tích thừa số) | `O(sqrt c)` trường hợp xấu | `O(1)` |

Với `c = 2^31 - 1`: vét cạn cỡ `10^9` phép; ba cách `O(sqrt c)` cỡ `3 * 10^4` đến `5 * 10^4` phép. Đo thực tế trên các `c` lớn:

In [23]:
tests = {
    "nguyên tố 4k+3 lớn nhất (2^31-1)": 2**31 - 1,
    "số chính phương lớn (46340^2)": 46340 ** 2,
    "tổng hai bình phương lớn": 30000 ** 2 + 30000 ** 2,
    "số lớn gần giới hạn": 2147483629,
}
solvers_fast = [("cách 2", Solution2().judgeSquareSum), ("cách 3", Solution3().judgeSquareSum),
                ("cách 4", Solution4().judgeSquareSum), ("cách 5", Solution5().judgeSquareSum)]
for name, c in tests.items():
    row, answers = [], set()
    for label, f in solvers_fast:
        t0 = time.perf_counter()
        r = f(c)
        ms = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{label} {ms:6.1f} ms")
    assert len(answers) == 1
    print(f"c = {c:>10} ({name}): {answers.pop()!s:5} | " + " | ".join(row))

c = 2147483647 (nguyên tố 4k+3 lớn nhất (2^31-1)): False | cách 2    6.9 ms | cách 3    6.4 ms | cách 4  109.4 ms | cách 5   10.2 ms
c = 2147395600 (số chính phương lớn (46340^2)): True  | cách 2    0.0 ms | cách 3    0.0 ms | cách 4    0.0 ms | cách 5    0.1 ms


c = 1800000000 (tổng hai bình phương lớn): True  | cách 2    1.8 ms | cách 3    1.2 ms | cách 4   27.7 ms | cách 5    0.0 ms


c = 2147483629 (số lớn gần giới hạn): True  | cách 2    3.9 ms | cách 3    2.7 ms | cách 4   36.6 ms | cách 5    7.0 ms


Mọi cách `O(sqrt c)` đều chạy trong vài chục mili-giây. (Số `2147483629` chỉ là một số lớn gần giới hạn để thử tốc độ; ở mỗi dòng ta đã assert các cách cho cùng đáp án.)

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta viết `check_all` chạy mọi cách trên cùng `c` và đòi tất cả trùng nhau; vét cạn chỉ dùng cho `c` nhỏ.

In [24]:
def oracle_set(c):
    """Oracle độc lập: tập các bình phương <= c, kiểm tra c - s có trong tập không."""
    squares = {i * i for i in range(isqrt(c) + 1)}
    return any((c - s) in squares for s in squares)


fast_solvers = [("cách 2", Solution2().judgeSquareSum), ("cách 3", Solution3().judgeSquareSum),
                ("cách 4", Solution4().judgeSquareSum), ("cách 5", Solution5().judgeSquareSum)]


def check_all(c, expected=None, brute=True):
    results = {name: f(c) for name, f in fast_solvers}
    if brute:
        results["vét cạn"] = judge_brute(c)
    values = set(results.values())
    assert len(values) == 1, (c, results)
    got = values.pop()
    if expected is not None:
        assert got == expected, (c, got, expected)
    return got


for c, expected in examples:
    check_all(c, expected)
print("ví dụ đề: OK")

edge = [
    (0, True),            # 0 = 0^2 + 0^2
    (1, True),            # 0 + 1
    (2, True),            # 1 + 1  (a == b)
    (3, False),
    (4, True),            # 0 + 4 (a = 0)
    (6, False),
    (7, False),
    (8, True),            # 2^2 + 2^2
    (9, True),            # 0 + 9
    (10, True),           # 1 + 9
    (11, False),
    (12, False),          # 2^2 * 3: 3 mũ lẻ
    (13, True),           # 4 + 9
    (25, True),           # 0 + 25 và 9 + 16
    (45, True),           # 3^2 * 5
    (49, True),           # 0 + 49 (7 mũ chẵn)
    (21, False),
    (99, False),
]
for c, expected in edge:
    check_all(c, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


### 12.2. Phản ví dụ cho các suy nghĩ sai

Một vài "tối ưu" nghe có lý nhưng sai:

In [25]:
def wrong_strict(c):
    """SAI: dùng a < b (loại trường hợp a == b)."""
    a, b = 0, isqrt(c)
    while a < b:
        s = a * a + b * b
        if s == c:
            return True
        if s < c:
            a += 1
        else:
            b -= 1
    return False


def wrong_only_prime_mod(c):
    """SAI: 'c có ước nguyên tố 4k+3 thì không biểu diễn được' (bỏ qua số mũ chẵn)."""
    for p, e in factorize(c).items():
        if p % 4 == 3:
            return False
    return True


def wrong_even_c(c):
    """SAI: 'số chẵn thì luôn biểu diễn được'."""
    return c % 2 == 0


for c, why in ((2, "a == b"), (0, "a == b == 0"), (9, "3 mũ chẵn"), (12, "chẵn nhưng 3 mũ lẻ")):
    print(f"c = {c:>2} ({why:20}): đúng = {oracle_set(c)!s:5} | a<b = {wrong_strict(c)!s:5} | bỏ mũ = {wrong_only_prime_mod(c)!s:5} | chẵn = {wrong_even_c(c)!s:5}")

assert wrong_strict(2) != oracle_set(2)
assert wrong_only_prime_mod(9) != oracle_set(9)
assert wrong_even_c(12) != oracle_set(12)

c =  2 (a == b              ): đúng = True  | a<b = False | bỏ mũ = True  | chẵn = True 
c =  0 (a == b == 0         ): đúng = True  | a<b = False | bỏ mũ = True  | chẵn = True 
c =  9 (3 mũ chẵn           ): đúng = True  | a<b = True  | bỏ mũ = False | chẵn = False
c = 12 (chẵn nhưng 3 mũ lẻ  ): đúng = False | a<b = False | bỏ mũ = False | chẵn = True 


- `c = 2`: chỉ có `1^2 + 1^2`; dùng `a < b` bỏ lỡ.
- `c = 9`: `9 = 3^2 + 0^2` đúng, nhưng "có ước `3` thì loại" sai vì `3` có mũ chẵn.
- `c = 12`: là số chẵn nhưng `3` có mũ lẻ, không biểu diễn được.

### 12.3. Đối chiếu nhiều input (vét cạn toàn bộ `c` nhỏ + ngẫu nhiên có seed)

Trước hết kiểm tra **mọi** `c` từ `0` đến `3000` với cả 5 cách (kể cả vét cạn và oracle tập hợp) — cũng là cách kiểm chứng định lý Fermat bằng thực nghiệm.

In [26]:
true_count = 0
for c in range(0, 3001):
    got = check_all(c, brute=True)
    assert got == oracle_set(c), c
    true_count += got
print(f"mọi c trong [0, 3000]: 5 cách + oracle đồng ý; có {true_count} số biểu diễn được")
print("20 số biểu diễn được đầu tiên:", [c for c in range(60) if oracle_set(c)][:20])

# kiểm chứng định lý Fermat bằng thực nghiệm cho c đến 40000 (chỉ so cách 5 với oracle)
for c in range(3001, 40001):
    assert Solution5().judgeSquareSum(c) == oracle_set(c), c
print("cách 5 (Fermat) khớp oracle với mọi c trong [3001, 40000]")

mọi c trong [0, 3000]: 5 cách + oracle đồng ý; có 900 số biểu diễn được
20 số biểu diễn được đầu tiên: [0, 1, 2, 4, 5, 8, 9, 10, 13, 16, 17, 18, 20, 25, 26, 29, 32, 34, 36, 37]


cách 5 (Fermat) khớp oracle với mọi c trong [3001, 40000]


In [27]:
rng = random.Random(20633)
count_true = 0
for _ in range(400):
    c = rng.randint(0, 2 * 10**6)
    got = check_all(c, brute=False)
    assert got == oracle_set(c)
    count_true += got
print(f"400 c ngẫu nhiên trong [0, 2*10^6]: các cách đồng ý với oracle ({count_true} True)")

# c lớn trong ràng buộc: đối chiếu 4 cách nhanh với nhau (không còn vét cạn)
for _ in range(40):
    c = rng.randint(0, 2**31 - 1)
    vals = {f(c) for _, f in fast_solvers}
    assert len(vals) == 1, c

# các c cực biên / đặc biệt
assert Solution().judgeSquareSum(2**31 - 1) is False          # nguyên tố 3 mod 4
assert Solution().judgeSquareSum(46340 ** 2) is True          # số chính phương lớn nhất <= 2^31-1
assert Solution().judgeSquareSum(46340 ** 2 + 1) is True      # 46340^2 + 1^2
assert Solution().judgeSquareSum(10 ** 9) is True             # 2^9 * 5^9
print("c lớn và cực biên: OK")

400 c ngẫu nhiên trong [0, 2*10^6]: các cách đồng ý với oracle (89 True)


c lớn và cực biên: OK


Ngoài các giá trị đặc biệt, ta còn so cách nộp với oracle độc lập (tập `~46340` bình phương, vẫn nhanh) trên các `c` lớn ngẫu nhiên:

In [28]:
rng = random.Random(7)
for _ in range(25):
    c = rng.randint(0, 2**31 - 1)
    assert Solution().judgeSquareSum(c) == oracle_set(c), c
print("25 c lớn ngẫu nhiên: cách nộp khớp oracle độc lập")

25 c lớn ngẫu nhiên: cách nộp khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Loại trường hợp `a == b` (dùng `a < b`)

**Code sai:**

In [29]:
class Wrong1:
    def judgeSquareSum(self, c):
        a, b = 0, isqrt(c)
        while a < b:                    # SAI: phải là a <= b
            s = a * a + b * b
            if s == c:
                return True
            if s < c:
                a += 1
            else:
                b -= 1
        return False

for c in (2, 8, 18, 0):
    print(f"c = {c:>2}: sai = {Wrong1().judgeSquareSum(c)!s:5} | đúng = {Solution().judgeSquareSum(c)}")

c =  2: sai = False | đúng = True
c =  8: sai = False | đúng = True
c = 18: sai = False | đúng = True
c =  0: sai = False | đúng = True


**Vì sao sai:** `2 = 1^2 + 1^2`, `8 = 2^2 + 2^2`, `18 = 3^2 + 3^2`, `0 = 0^2 + 0^2` đều có `a == b`. Điều kiện vòng lặp phải cho phép hai con trỏ gặp nhau: `a <= b`.

### Lỗi 2: Quên `+ 1` ở cận của `range`

In [30]:
class Wrong2:
    def judgeSquareSum(self, c):
        for a in range(isqrt(c)):       # SAI: thiếu + 1 nên bỏ giá trị a = isqrt(c)
            rest = c - a * a
            b = isqrt(rest)
            if b * b == rest:
                return True
        return False

for c in (0, 2, 8, 1):
    print(f"c = {c}: sai = {Wrong2().judgeSquareSum(c)!s:5} | đúng = {Solution().judgeSquareSum(c)}")

c = 0: sai = False | đúng = True
c = 2: sai = False | đúng = True
c = 8: sai = False | đúng = True
c = 1: sai = True  | đúng = True


**Vì sao sai:** `range(n)` chạy `0..n-1`, không gồm `n`. Những `c` có đáp án với `a = 0` (như `c = 1`) vẫn "may mắn" đúng, nhưng `c = 0` làm `range(0)` rỗng, còn `c = 2` và `c = 8` cần `a = 1` và `a = 2` (bằng `isqrt(c)`) nên bị bỏ lỡ. Hãy dùng `range(isqrt(c) + 1)` hoặc điều kiện `while` rõ ràng.

### Lỗi 3: Vét cạn hai vòng lặp bị TLE

In [31]:
def slow_two_loops(c):
    r = isqrt(c)
    for a in range(r + 1):
        for b in range(r + 1):          # CHẬM: O(c)
            if a * a + b * b == c:
                return True
    return False

t0 = time.perf_counter()
slow_two_loops(99_999)                  # c nhỏ mà đã chậm
t1 = time.perf_counter() - t0
t0 = time.perf_counter()
Solution().judgeSquareSum(99_999)
t2 = time.perf_counter() - t0
print(f"c = 99999: hai vòng lặp {t1 * 1000:8.1f} ms | hai con trỏ {t2 * 1000:8.3f} ms")
print("với c = 2^31 - 1 hai vòng lặp cần ~", (2**31 - 1) // 1, "phép thử (~ 2 tỉ)")

c = 99999: hai vòng lặp      5.9 ms | hai con trỏ    0.065 ms
với c = 2^31 - 1 hai vòng lặp cần ~ 2147483647 phép thử (~ 2 tỉ)


**Vì sao sai:** `c` tới `2^31 - 1` nên `O(c)` là quá chậm; ta phải bỏ vòng lặp trong cùng (tính `b` từ `a`).

### Lỗi 4: Fermat — quên nguyên tố còn dư sau vòng chia thử

In [32]:
class Wrong4:
    def judgeSquareSum(self, c):
        i = 2
        while i * i <= c:
            if c % i == 0:
                e = 0
                while c % i == 0:
                    c //= i
                    e += 1
                if i % 4 == 3 and e % 2 == 1:
                    return False
            i += 1
        return True                     # SAI: quên xét c còn dư (nguyên tố lớn)

for c in (7, 3, 21, 11, 5):
    print(f"c = {c:>2}: sai = {Wrong4().judgeSquareSum(c)!s:5} | đúng = {Solution5().judgeSquareSum(c)}")

c =  7: sai = True  | đúng = False
c =  3: sai = True  | đúng = False
c = 21: sai = False | đúng = False
c = 11: sai = True  | đúng = False
c =  5: sai = True  | đúng = True


**Vì sao sai:** với `c = 7`, vòng `i = 2` rồi `i * i = 9 > 7` dừng, chưa chia được gì; `7` là nguyên tố `≡ 3 (mod 4)` mũ 1 nên phải `False`. Nguyên tố lớn nhất của `c` có thể **không** bị vòng chia thử xử lý và còn lại ở biến `c`; phải kiểm tra `c % 4 != 3` ở cuối.

### Lỗi 5: Fermat — loại mọi nguyên tố `4k + 3` bất kể số mũ
Đã minh họa ở mục 12.2 (`wrong_only_prime_mod`): `9 = 3^2` hợp lệ vì `3` có **mũ chẵn**. Điều kiện đúng là `p % 4 == 3 and e % 2 == 1` mới loại.

In [33]:
print("9  :", wrong_only_prime_mod(9), "(sai) vs", Solution5().judgeSquareSum(9), "(đúng)")
print("45 :", wrong_only_prime_mod(45), "(sai) vs", Solution5().judgeSquareSum(45), "(đúng)")
print("81 :", wrong_only_prime_mod(81), "(sai) vs", Solution5().judgeSquareSum(81), "(đúng)")

9  : False (sai) vs True (đúng)
45 : False (sai) vs True (đúng)
81 : False (sai) vs True (đúng)


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "có hai số sao cho **tổng bình phương** (hoặc tổng/hiệu/tích) bằng `c`" → cố định một số, **số còn lại được xác định**; hoặc hai con trỏ trên dãy đơn điệu.
- **Giảm một chiều:** mọi khi có hai biến bị ràng buộc bởi một phương trình, hãy tìm cách chỉ duyệt **một** biến (`O(n^2)` → `O(n)`), như Two Sum.
- **Dùng tính đối xứng:** `a^2 + b^2` đối xứng nên giả sử `a <= b` để giảm một nửa phạm vi và tránh duyệt trùng.
- **Hai con trỏ** dùng được khi hàm `f(a) + g(b)` đơn điệu theo từng con trỏ (ở đây `a^2` tăng theo `a`, `b^2` giảm khi giảm `b`).
- **Kiểm tra số chính phương:** luôn dùng `isqrt(x) ** 2 == x`, không dùng `sqrt` thực.
- **Toán học:** biết định lý tổng hai bình phương cho phép giải bằng phân tích thừa số. Các bài "tổng bốn bình phương" (LeetCode 279, Lagrange) cũng thuộc họ này.
- **Python:** số nguyên không tràn; `isqrt` chính xác; `while` với điều kiện kết hợp tốt cho hai con trỏ.
- **Test:** dùng oracle độc lập (tập các bình phương) và so với **mọi** `c` nhỏ — đây là cách tin tưởng nhất khi thuật toán dựa vào một định lý.

## 15. Biến thể

1. **Trả về cặp `(a, b)`** (hoặc `None`) thay vì chỉ `True/False` (demo bên dưới).
2. **Đếm số cách biểu diễn** `c = a^2 + b^2` với `0 <= a <= b` (hai con trỏ không dừng khi gặp đáp án mà tiếp tục).
3. **LeetCode 279 — Perfect Squares:** số bình phương ít nhất cộng thành `n` (Lagrange: luôn `<= 4`; có thể quy về kiểm tra 1, 2, 3 hoặc 4).
4. **LeetCode 367 — Valid Perfect Square:** chỉ kiểm tra số chính phương (cách 4 là phần con của nó).
5. **Tổng hai lập phương:** `a^3 + b^3 = c` — cùng khung hai con trỏ/cố định một biến, nhưng không có định lý đẹp như Fermat.
6. **Số nguyên Gauss:** `a^2 + b^2 = (a + bi)(a - bi)` — lý do sâu xa của định lý Fermat.

Demo biến thể 1, 2 (đối chiếu với oracle):

In [34]:
def find_pair(c):
    a, b = 0, isqrt(c)
    while a <= b:
        s = a * a + b * b
        if s == c:
            return (a, b)
        if s < c:
            a += 1
        else:
            b -= 1
    return None


def count_representations(c):
    """Số cặp (a, b) với 0 <= a <= b và a^2 + b^2 = c."""
    a, b, cnt = 0, isqrt(c), 0
    while a <= b:
        s = a * a + b * b
        if s == c:
            cnt += 1
            a += 1
            b -= 1
        elif s < c:
            a += 1
        else:
            b -= 1
    return cnt


for c in (5, 25, 50, 65, 3, 0, 325):
    print(f"c = {c:>3}: một cặp = {find_pair(c)} | số cách = {count_representations(c)}")

for c in range(0, 2000):
    brute = sum(1 for a in range(isqrt(c) + 1) for b in range(a, isqrt(c) + 1) if a * a + b * b == c)
    assert count_representations(c) == brute
    p = find_pair(c)
    assert (p is not None) == oracle_set(c)
    if p:
        assert p[0] ** 2 + p[1] ** 2 == c
print("biến thể OK")

c =   5: một cặp = (1, 2) | số cách = 1
c =  25: một cặp = (0, 5) | số cách = 2
c =  50: một cặp = (1, 7) | số cách = 2
c =  65: một cặp = (1, 8) | số cách = 2
c =   3: một cặp = None | số cách = 0
c =   0: một cặp = (0, 0) | số cách = 1
c = 325: một cặp = (1, 18) | số cách = 3
biến thể OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi cặp `(a, b)` | `O(c)` | `O(1)` | Chỉ kiểm chứng `c` nhỏ |
| 2. Duyệt `a` + `isqrt` | `b` xác định từ `a`; `a <= sqrt(c/2)` | `O(sqrt c)` | `O(1)` | Dễ hiểu nhất, ngắn |
| 3. Hai con trỏ | `a` tăng khi thiếu, `b` giảm khi dư | `O(sqrt c)` | `O(1)` | **Cách nộp**, không cần `isqrt` cho từng `a` |
| 4. Duyệt `a` + tìm nhị phân | Thay `isqrt` bằng nhị phân | `O(sqrt(c) log c)` | `O(1)` | Ngôn ngữ không có `isqrt` chính xác |
| 5. Fermat | Mọi nguyên tố `4k+3` có mũ chẵn | `O(sqrt c)` xấu nhất | `O(1)` | Khi `c` có nhiều ước nhỏ; hay khi cần chứng minh toán học |

Khác nhau thật về ý tưởng: cách 1 không có cấu trúc; cách 2 đổi một vòng lặp thành một phép tra `isqrt`; cách 3 dựa vào tính đơn điệu của tổng; cách 4 dựa vào tìm kiếm nhị phân; cách 5 dùng **lý thuyết số** để không cần tìm `a, b` chút nào.

In [35]:
# so sánh thời gian trên nhiều c lớn ngẫu nhiên (chạy cùng một tập c cho mọi cách)
rng = random.Random(16)
cs = [rng.randint(2**30, 2**31 - 1) for _ in range(30)]
ref = [oracle_set(c) for c in cs]
for name, f in fast_solvers:
    t0 = time.perf_counter()
    got = [f(c) for c in cs]
    ms = (time.perf_counter() - t0) * 1000
    assert got == ref
    print(f"{name}: tổng {ms:8.1f} ms cho 30 giá trị c | số True: {sum(got)}")

cách 2: tổng    217.5 ms cho 30 giá trị c | số True: 6
cách 3: tổng    107.5 ms cho 30 giá trị c | số True: 6


cách 4: tổng   1350.4 ms cho 30 giá trị c | số True: 6
cách 5: tổng      4.9 ms cho 30 giá trị c | số True: 6


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [36]:
print(Solution().judgeSquareSum(0))
print(Solution().judgeSquareSum(18))
print(Solution().judgeSquareSum(14))
print(Solution().judgeSquareSum(1_000_000))

True
True
False
True


**Bài 2 (tính tay).** (a) Liệt kê **mọi** cặp `(a, b)` với `0 <= a <= b` và `a^2 + b^2 = 50`; gán vào `ans2a` (list các tuple, đang là `None`). (b) Dùng định lý Fermat, `c = 252 = 2^2 * 3^2 * 7` có biểu diễn được không? Gán `ans2b` (True/False, đang là `None`).

In [37]:
ans2a = None   # TODO: list các tuple (a, b)
ans2b = None   # TODO: True hoặc False

**Bài 3 (điền TODO).** Hoàn thành `is_perfect_square(x)` bằng **tìm nhị phân** (không dùng `isqrt`, không dùng `**0.5`).

In [38]:
def is_perfect_square(x):
    # TODO: lo, hi = 0, x ; tìm mid sao cho mid * mid == x
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_judge(c)` theo cách hai con trỏ (cách 3).

In [39]:
def my_judge(c):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `next_sum_of_two_squares(c)`: số nhỏ nhất `c' >= c` biểu diễn được dưới dạng `a^2 + b^2`. (b) Viết `closest_gap(c)`: `min |a^2 + b^2 - c|` trên mọi `a, b` nguyên không âm (khoảng cách nhỏ nhất từ `c` đến một tổng hai bình phương; bằng `0` khi `c` biểu diễn được). Ví dụ `c = 3` → `next = 4`, `closest_gap = 1` (vì `2 = 1 + 1` và `4 = 0 + 4` cùng cách 3 một đơn vị).

In [40]:
def next_sum_of_two_squares(c):
    # TODO
    pass

def closest_gap(c):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: thử từng số: `0 = 0 + 0`; `18 = 9 + 9`; `14` có các bình phương `0, 1, 4, 9`.
- Bài 2: (a) các bình phương `<= 50` là `0, 1, 4, 9, 16, 25, 36, 49`; tìm hai số cộng ra 50. (b) nguyên tố `7 ≡ 3 (mod 4)` mũ mấy?
- Bài 3: nhị phân trên `[0, x]` như ở mục 3.5.
- Bài 4: `a = 0`, `b = isqrt(c)`; `while a <= b`.
- Bài 5: lặp `c' = c, c + 1, ...` và dùng `Solution` để kiểm tra.

**Mức 2 (vừa):**
- Bài 2: (a) `50 = 1 + 49 = 25 + 25` — kiểm tra thêm các cặp khác. (b) `7` có mũ `1` (lẻ).
- Bài 3: nếu `mid * mid < x` thì `lo = mid + 1`, nếu `>` thì `hi = mid - 1`.
- Bài 4: `s < c` → `a += 1`; `s > c` → `b -= 1`.
- Bài 5(b): nếu `c` không biểu diễn được thì khoảng cách nhỏ nhất là `min(c - prev, next - c)` với `prev` là số biểu diễn được lớn nhất `< c`. Nhớ `0 = 0^2 + 0^2` là biểu diễn được nên `prev` luôn tồn tại với `c >= 1`.

**Mức 3 (gần đáp án):**
- Bài 3: `lo, hi = 0, x; while lo <= hi: mid = (lo + hi) // 2; ...; return False`.
- Bài 4: xem mục 9, code nộp.
- Bài 5: (a) `while not Solution().judgeSquareSum(c): c += 1; return c`. (b) tìm `prev` bằng vòng lặp lùi `p = c; while not judge(p): p -= 1` và `next` như (a), rồi lấy `min`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [41]:
# Bài 1
assert Solution().judgeSquareSum(0) is True
assert Solution().judgeSquareSum(18) is True
assert Solution().judgeSquareSum(14) is False
assert Solution().judgeSquareSum(1_000_000) is True       # 1000^2 + 0^2

# Bài 2
ans2a = [(1, 7), (5, 5)]
ans2b = False
assert ans2a == [(a, b) for a in range(8) for b in range(a, 8) if a * a + b * b == 50]
assert ans2b == Solution5().judgeSquareSum(252) == oracle_set(252)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [42]:
# Bài 3
def is_perfect_square(x):
    lo, hi = 0, x
    while lo <= hi:
        mid = (lo + hi) // 2
        sq = mid * mid
        if sq == x:
            return True
        if sq < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

for x in range(0, 5000):
    assert is_perfect_square(x) == (isqrt(x) ** 2 == x), x
assert is_perfect_square(46340 ** 2) and not is_perfect_square(46340 ** 2 + 1)
print("Bài 3 OK")

Bài 3 OK


In [43]:
# Bài 4
def my_judge(c):
    a, b = 0, isqrt(c)
    while a <= b:
        s = a * a + b * b
        if s == c:
            return True
        if s < c:
            a += 1
        else:
            b -= 1
    return False

for c in range(0, 3000):
    assert my_judge(c) == oracle_set(c), c
rng = random.Random(4)
for _ in range(50):
    c = rng.randint(0, 2**31 - 1)
    assert my_judge(c) == Solution().judgeSquareSum(c)
print("Bài 4 OK")

Bài 4 OK


In [44]:
# Bài 5
def next_sum_of_two_squares(c):
    while not Solution().judgeSquareSum(c):
        c += 1
    return c


def closest_gap(c):
    nxt = next_sum_of_two_squares(c)
    p = c
    while not Solution().judgeSquareSum(p):
        p -= 1                        # 0 = 0^2 + 0^2 luôn biểu diễn được nên dừng được
    return min(c - p, nxt - c)


assert next_sum_of_two_squares(3) == 4
assert closest_gap(3) == 1
assert next_sum_of_two_squares(5) == 5 and closest_gap(5) == 0
assert next_sum_of_two_squares(6) == 8 and closest_gap(6) == 1     # 5 và 8: gần nhất là 5, cách 1
assert next_sum_of_two_squares(7) == 8 and closest_gap(7) == 1
assert next_sum_of_two_squares(0) == 0 and closest_gap(0) == 0

# đối chiếu với vét cạn: min |a^2 + b^2 - c| trên miền đủ lớn
for c in range(0, 400):
    r = isqrt(c) + 2
    brute = min(abs(a * a + b * b - c) for a in range(r + 1) for b in range(r + 1))
    assert closest_gap(c) == brute, c
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template hai con trỏ (nộp):**

```python
from math import isqrt

class Solution:
    def judgeSquareSum(self, c: int) -> bool:
        a, b = 0, isqrt(c)
        while a <= b:                 # cho phép a == b
            s = a * a + b * b
            if s == c:
                return True
            if s < c:
                a += 1                # còn thiếu -> tăng số nhỏ
            else:
                b -= 1                # dư -> giảm số lớn
        return False
```

**Template Fermat:**

```python
i = 2
while i * i <= c:
    if c % i == 0:
        e = 0
        while c % i == 0:
            c //= i
            e += 1
        if i % 4 == 3 and e % 2 == 1:
            return False
    i += 1
return c % 4 != 3
```

**Checklist trước khi submit:**

- [ ] Cho phép `a == b` (`while a <= b`) và `a = 0` (đã thử `c = 0, 1, 2, 4`)?
- [ ] Dùng `isqrt` (chính xác), không so sánh bằng số thực?
- [ ] Cận trên của `b` là `isqrt(c)`, của `a` (cách 2) là `isqrt(c // 2)` — có `+ 1` đúng ở `range` chưa?
- [ ] Độ phức tạp `O(sqrt c)` (không phải `O(c)`) cho `c` tới `2^31 - 1`?
- [ ] Nếu dùng Fermat: xử lý nguyên tố còn dư sau vòng chia thử, và chỉ loại khi **mũ lẻ**?
- [ ] Đã so với oracle độc lập (tập bình phương) trên mọi `c` nhỏ?